# 02. Feature Checks

Check that the card-level feature table is ready for modeling.

In [ ]:
import numpy as np
import pandas as pd

from mdq.data import PROJECT_ROOT

pd.options.display.float_format = "{:,.4f}".format

FEATURE_PATH = PROJECT_ROOT / "data" / "processed" / "card_features.parquet"
features = pd.read_parquet(FEATURE_PATH, engine="fastparquet")

features.shape

In [ ]:
row_checks = pd.Series({
    "rows_total": len(features),
    "business_rows": (features["segment"] == "business").sum(),
    "consumer_rows": (features["segment"] == "consumer").sum(),
    "duplicate_cards": features["card_number"].duplicated().sum(),
    "missing_values": features.isna().sum().sum(),
    "infinite_values": np.isinf(features.select_dtypes(include="number")).sum().sum(),
})

row_checks

In [ ]:
ratio_cols = [c for c in features.columns if c.endswith("_ratio") or c.endswith("_share")]

ratio_checks = pd.DataFrame({
    "min": features[ratio_cols].min(),
    "max": features[ratio_cols].max(),
})

ratio_checks["valid_0_1"] = ratio_checks["min"].ge(0) & ratio_checks["max"].le(1)
ratio_checks

In [ ]:
non_negative_cols = [
    "txn_count",
    "amount_cv",
    "log_total_spend",
    "log_avg_amount",
    "log_median_amount",
    "log_std_amount",
    "log_p90_amount",
    "active_days",
    "txn_per_active_day",
    "log_amount_per_active_day",
    "unique_merchants",
    "unique_mcc",
    "merchant_entropy",
    "mcc_entropy",
    "top_merchant_share",
    "top_mcc_share",
]

pd.DataFrame({
    "min": features[non_negative_cols].min(),
    "valid_non_negative": features[non_negative_cols].min().ge(0),
})

In [ ]:
id_cols = ["card_number", "segment", "label"]
raw_money_cols = ["total_spend", "avg_amount", "median_amount", "std_amount", "max_amount", "p90_amount", "amount_per_active_day"]
removed_feature_cols = ["large_txn_10000_ratio", "weekday_non_business_hours_ratio", "monthly_spend_growth", "log_max_amount"]
dropped_model_cols = id_cols + ["pos_ratio", "active_months"]
model_feature_cols = [c for c in features.columns if c not in dropped_model_cols]

pd.Series({
    "model_feature_count": len(model_feature_cols),
    "raw_money_cols_present": sorted(set(raw_money_cols) & set(features.columns)),
    "removed_feature_cols_present": sorted(set(removed_feature_cols) & set(features.columns)),
    "pos_ratio_in_model": "pos_ratio" in model_feature_cols,
    "active_months_in_model": "active_months" in model_feature_cols,
    "forbidden_columns_present": sorted(set(["merchant_id", "merchant_name", "mcc_name"]) & set(features.columns)),
})

In [ ]:
summary = (
    features.groupby("segment")[model_feature_cols]
    .median()
    .T
    .assign(diff=lambda x: x["business"] - x["consumer"])
    .sort_values("diff", ascending=False)
)

summary.head(20)

## Notes

`card_number`, `segment`, and `label` are kept for tracking and validation, but they should not be included in model input `X`.

For the first model, use log money features instead of raw money features. After the first model, check correlation and feature importance, then remove weak or duplicated columns.